# AI Search Configuraiton & Setup

### Pulling data into an index from blob storage requires the following to be setup:

- Data Source
- Index
- Skillset
- Indexer

The pull model uses indexers connecting to a supported data source, automatically uploading the data into your index.

Note to leverage this notebook requires:

- Install .NET 8
- Launch VS Code and install "Polyglot" extention

## AI Search Index Creation


In [89]:
#r "nuget: dotenv.net, 3.0.0"
#r "nuget: Newtonsoft.Json"
#r "nuget: Azure.AI.OpenAI"
#r "nuget: Azure.Core"
#r "nuget: Azure.Identity"
#r "nuget: Newtonsoft.Json.Schema"

Installed Packages Azure.AI.OpenAI, 2.1.0 Azure.Core, 1.45.0 Azure.Identity, 1.13.2 dotenv.net, 3.0.0 Newtonsoft.Json, 13.0.3 Newtonsoft.Json.Schema, 4.0.1

In [90]:
using System;  
using System.IO;  
using System.Net.Http;  
using System.Text;  
using System.Threading.Tasks;  
using Newtonsoft.Json; 
  
public static void Load(string filePath)  
{  
    if (!File.Exists(filePath))  
        return;  
  
    foreach (var line in File.ReadAllLines(filePath))  
    {  
        // Split the line into exactly two parts  
        var parts = line.Split(new char[] {'='}, 2);  
  
        if (parts.Length != 2)  
            continue;  // Skip lines that don't properly split into two parts  
  
        // Trim parts to remove any accidental whitespace  
        string key = parts[0].Trim();  
        string value = parts[1].Trim();  
  
        Environment.SetEnvironmentVariable(key, value);  
  
        // Optional: Add a debug line to confirm what's being set  
        //Console.WriteLine($"Setting {key} to {value}");  
    }  
}  

public static void CheckEmpty(string variableName)  
{  
    // Retrieve the value of the environment variable  
    string value = Environment.GetEnvironmentVariable(variableName);  
  
    // Check if the value is null or empty  
    if (string.IsNullOrEmpty(value))  
    {  
        Console.WriteLine($"{variableName} is empty or not set.");  
    }  
    // else  
    // {  
    //     Console.WriteLine($"{variableName} is set to: {value}");  
    // }  
}  

In [91]:
Load(".env");

// Check for empty environment variables  
CheckEmpty("AZURE_SEARCH_SERVICE_NAME");  
CheckEmpty("AZURE_SEARCH_INDEX");  
CheckEmpty("AZURE_SEARCH_API_KEY");  
CheckEmpty("BLOB_CONNECTION_STRING");  
CheckEmpty("BLOB_CONTAINER_NAME");  
CheckEmpty("AZURE_OPENAI_ENDPOINT");  
CheckEmpty("AZURE_OPENAI_EMBEDDING_MODEL_NAME");  
CheckEmpty("AZURE_OPENAI_EMBEDDING_DIMENSIONS");  
CheckEmpty("AZURE_AI_SERVICES_ENDPOINT"); 

string azureSearchServiceName = Environment.GetEnvironmentVariable("AZURE_SEARCH_SERVICE_NAME");  
string azureSearchIndex = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX");  
string azureSearchIndexLayout = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX_LAYOUT");  
string azureSearchIndexOcr = Environment.GetEnvironmentVariable("AZURE_SEARCH_INDEX_OCR");  
string azureSearchApiKey = Environment.GetEnvironmentVariable("AZURE_SEARCH_API_KEY");  
string blobConnectionString = Environment.GetEnvironmentVariable("BLOB_CONNECTION_STRING")?.Trim('"'); 
string blobContainerName = Environment.GetEnvironmentVariable("BLOB_CONTAINER_NAME");  
string azureOpenAiEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");  
string azureOpenAiEmbeddingModelName = Environment.GetEnvironmentVariable("AZURE_OPENAI_EMBEDDING_MODEL_NAME");  
string azureOpenAiEmbeddingDimensions = Environment.GetEnvironmentVariable("AZURE_OPENAI_EMBEDDING_DIMENSIONS");  
string azureAiServicesEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_SERVICES_ENDPOINT");  
  

Console.WriteLine("Variables have been set");

Variables have been set


## Create Data Source

https://learn.microsoft.com/en-us/azure/search/search-howto-indexing-azure-blob-storage

In [92]:
public static void CreateDataSource(string serviceName, string indexName, string searchApiKey, string storageConnectionString, string storageContainer)  
{  
    Console.WriteLine("AZURE_SEARCH_INDEX = {0}", indexName);  
    Console.WriteLine("index_name = {0}", indexName);  

    string endpoint = $"https://{serviceName}.search.windows.net/";  
    string url = $"{endpoint}/datasources/{indexName}-datasource?api-version=2024-11-01-preview";  

    var payload = $@"  
    {{  
        ""name"": ""{indexName}-datasource"",  
        ""description"": null,  
        ""type"": ""azureblob"",  
        ""subtype"": null,  
        ""credentials"": {{  
            ""connectionString"": ""{storageConnectionString}""  
        }},  
        ""container"": {{  
            ""name"": ""{storageContainer}"",  
            ""query"": null  
        }},  
        ""dataChangeDetectionPolicy"": null,  
        ""dataDeletionDetectionPolicy"": {{  
            ""@odata.type"": ""#Microsoft.Azure.Search.NativeBlobSoftDeleteDeletionDetectionPolicy""  
        }},  
        ""encryptionKey"": null,  
        ""identity"": null  
    }}";  

    using (var client = new HttpClient())  
    {  
        client.DefaultRequestHeaders.Add("api-key", searchApiKey);  
        var content = new StringContent(payload, Encoding.UTF8, "application/json");  

        HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
        Console.WriteLine("Response: {0}", response); // Print the response
        Console.WriteLine("HTTP Status Code: {0}", response.StatusCode); // Print the status code  

        if (!response.IsSuccessStatusCode)  
        {  
            Console.WriteLine("Error creating data source: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
        }  
    }  
}  

In [93]:
CreateDataSource(azureSearchServiceName, azureSearchIndex, azureSearchApiKey, blobConnectionString, blobContainerName); 

AZURE_SEARCH_INDEX = vector-manual31
index_name = vector-manual31
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766FB29942D2"
  Location: https://mmchrdemo03search.search.windows.net:443/datasources('vector-manual31-datasource')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: 07342d99-9442-472b-bda7-6234b118e7c0
  elapsed-time: 88
  Date: Tue, 08 Apr 2025 07:33:46 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


## Create Search 

https://learn.microsoft.com/en-us/azure/search/search-get-started-portal

https://learn.microsoft.com/en-us/azure/search/vector-search-ranking

For HNSW, try different levels of efConstruction to change the internal composition of the proximity graph. The default is 400. The range is 100 to 1,000.

In [94]:
public static void CreateIndexSemantic(string serviceName, string index, string azureSearchApiKey, string openaiApiBase, string textEmbbedingModel, int textEmbbedingModelDimensions)  
    {  
        string endpoint = $"https://{serviceName}.search.windows.net/";  
        string url = $"{endpoint}/indexes/{index}/?api-version=2024-11-01-preview";  
        Console.WriteLine(url);  
  
        var payload = $@"  
        {{  
            ""name"": ""{index}"",  
            ""fields"": [  
                {{""name"": ""chunk_id"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": true, ""facetable"": false, ""key"": true, ""analyzer"": ""keyword""}},  
                {{""name"": ""parent_id"", ""type"": ""Edm.String"", ""searchable"": false, ""filterable"": true, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""chunk"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""title"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""header_1"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""header_2"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""header_3"", ""type"": ""Edm.String"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""loads"", ""type"": ""Collection(Edm.String)"", ""searchable"": true, ""filterable"": true, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false}},  
                {{""name"": ""text_vector"", ""type"": ""Collection(Edm.Single)"", ""searchable"": true, ""filterable"": false, ""retrievable"": true, ""stored"": true, ""sortable"": false, ""facetable"": false, ""key"": false, ""dimensions"": {textEmbbedingModelDimensions}, ""vectorSearchProfile"": ""{index}-azureOpenAi-text-profile""}}  
            ],  
            ""scoringProfiles"": [],  
            ""suggesters"": [],  
            ""analyzers"": [],  
            ""normalizers"": [],  
            ""tokenizers"": [],  
            ""tokenFilters"": [],  
            ""charFilters"": [],  
            ""similarity"": {{  
                ""@odata.type"": ""#Microsoft.Azure.Search.BM25Similarity""  
            }},  
            ""semantic"": {{  
                ""configurations"": [  
                    {{  
                        ""name"": ""{index}-semantic-configuration"",  
                        ""prioritizedFields"": {{  
                            ""titleField"": {{""fieldName"": ""title""}},  
                            ""prioritizedContentFields"": [{{""fieldName"": ""chunk""}}]  
                        }}  
                    }}  
                ]  
            }},  
            ""vectorSearch"": {{  
                ""algorithms"": [  
                    {{  
                        ""name"": ""{index}-algorithm"",  
                        ""kind"": ""hnsw"",  
                        ""hnswParameters"": {{  
                            ""metric"": ""cosine"",  
                            ""m"": 4,  
                            ""efConstruction"": 400,  
                            ""efSearch"": 500  
                        }}  
                    }}  
                ],  
                ""profiles"": [  
                    {{  
                        ""name"": ""{index}-azureOpenAi-text-profile"",  
                        ""algorithm"": ""{index}-algorithm"",  
                        ""vectorizer"": ""{index}-azureOpenAi-text-vectorizer""  
                    }}  
                ],  
                ""vectorizers"": [  
                    {{  
                        ""name"": ""{index}-azureOpenAi-text-vectorizer"",  
                        ""kind"": ""azureOpenAI"",  
                        ""azureOpenAIParameters"": {{  
                            ""resourceUri"": ""{openaiApiBase}"",  
                            ""deploymentId"": ""{textEmbbedingModel}"",  
                            ""modelName"": ""{textEmbbedingModel}""  
                        }}  
                    }}  
                ],  
                ""compressions"": []  
            }}  
        }}";  
  
        //Console.WriteLine(payload);
        using (var client = new HttpClient())  
        {  
            client.DefaultRequestHeaders.Add("api-key", azureSearchApiKey);  
            var content = new StringContent(payload, Encoding.UTF8, "application/json");  
    
            HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
            Console.WriteLine("Response: {0}", response); // Print the response
            Console.WriteLine("HTTP Status Code: {0}", response.StatusCode); // Print the status code  
    
            if (!response.IsSuccessStatusCode)  
            {  
                Console.WriteLine("Error creating data source: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
            }  
        }  
    }  

In [95]:
CreateIndexSemantic(azureSearchServiceName, azureSearchIndex, azureSearchApiKey, azureOpenAiEndpoint, azureOpenAiEmbeddingModelName, int.Parse(azureOpenAiEmbeddingDimensions));

https://mmchrdemo03search.search.windows.net//indexes/vector-manual31/?api-version=2024-11-01-preview
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766FB3991839"
  Location: https://mmchrdemo03search.search.windows.net:443/indexes('vector-manual31')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: dedad811-3030-4b14-a0bd-0d7faccacb4d
  elapsed-time: 343
  Date: Tue, 08 Apr 2025 07:33:48 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


## Create a Skillset
- Layout Skill
    -  https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-document-intelligence-layout
- WebApi Skill
- Split Skill
    - https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-textsplit
- AzureOpenAIEmbeddingSkill
    - https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-azure-openai-embedding

In [96]:
public static void CreateSkillset(string serviceName, string searchApiKey, string index, string openaiApiBase, string textEmbeddingModel, string azureAiServicesEndpoint, string textEmbeddingModelDimensions)  
{  
    string endpoint = $"https://{serviceName}.search.windows.net/";  
    string resourceUri = openaiApiBase;  
    string deploymentId = textEmbeddingModel;  
    string modelName = textEmbeddingModel;  
    string dimensions = textEmbeddingModelDimensions;  
    string embeddingFunctionAppUriAndKey = Environment.GetEnvironmentVariable("AZURE_FUNCTION_APP_URL");  
    string azureAiServicesKey = Environment.GetEnvironmentVariable("AZURE_AI_SERVICES_KEY");  
  
    string url = $"{endpoint}/skillsets/{index}-skillset?api-version=2024-11-01-preview";  
    Console.WriteLine(url);  
  
    var payload = $@"  
    {{  
        ""name"": ""{index}-skillset"",  
        ""description"": ""Skillset to chunk documents and generate embeddings"",  
        ""skills"": [  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Util.DocumentIntelligenceLayoutSkill"",  
                ""name"": ""#1"",  
                ""context"": ""/document"",  
                ""outputMode"": ""oneToMany"",  
                ""markdownHeaderDepth"": ""h3"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""file_data"",  
                        ""source"": ""/document/file_data""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""markdown_document"",  
                        ""targetName"": ""markdownDocument""  
                    }}  
                ]  
            }},  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Custom.WebApiSkill"",  
                ""uri"": ""{embeddingFunctionAppUriAndKey}"",  
                ""httpMethod"": ""POST"",  
                ""timeout"": ""PT230S"",  
                ""batchSize"": 1,  
                ""degreeOfParallelism"": 1,  
                ""name"": ""LoadFinder"",  
                ""context"": ""/document"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""text"",  
                        ""source"": ""/document/markdownDocument""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""loads"",  
                        ""targetName"": ""loads""  
                    }}  
                ]  
            }},  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Text.SplitSkill"",  
                ""name"": ""#2"",  
                ""description"": ""Split skill to chunk documents"",  
                ""context"": ""/document/markdownDocument/*"",  
                ""defaultLanguageCode"": ""en"",  
                ""textSplitMode"": ""pages"",  
                ""maximumPageLength"": 2000,  
                ""pageOverlapLength"": 500,  
                ""maximumPagesToTake"": 0,  
                ""unit"": ""characters"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""text"",  
                        ""source"": ""/document/markdownDocument/*/content""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""textItems"",  
                        ""targetName"": ""pages""  
                    }}  
                ]  
            }},  
            {{  
                ""@odata.type"": ""#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill"",  
                ""name"": ""#3"",  
                ""context"": ""/document/markdownDocument/*/pages/*"",  
                ""resourceUri"": ""{resourceUri}"",  
                ""deploymentId"": ""{deploymentId}"",  
                ""dimensions"": ""{dimensions}"",  
                ""modelName"": ""{modelName}"",  
                ""inputs"": [  
                    {{  
                        ""name"": ""text"",  
                        ""source"": ""/document/markdownDocument/*/pages/*""  
                    }}  
                ],  
                ""outputs"": [  
                    {{  
                        ""name"": ""embedding"",  
                        ""targetName"": ""text_vector""  
                    }}  
                ]  
            }}  
        ],  
        ""cognitiveServices"": {{  
            ""@odata.type"": ""#Microsoft.Azure.Search.AIServicesByKey"",  
            ""subdomainUrl"": ""{azureAiServicesEndpoint}"",  
            ""key"": ""{azureAiServicesKey}""  
        }},  
        ""indexProjections"": {{  
            ""selectors"": [  
                {{  
                    ""targetIndexName"": ""{index}"",  
                    ""parentKeyFieldName"": ""parent_id"",  
                    ""sourceContext"": ""/document/markdownDocument/*/pages/*"",  
                    ""mappings"": [  
                        {{  
                            ""name"": ""text_vector"",  
                            ""source"": ""/document/markdownDocument/*/pages/*/text_vector""  
                        }},  
                        {{  
                            ""name"": ""chunk"",  
                            ""source"": ""/document/markdownDocument/*/pages/*""  
                        }},  
                        {{  
                            ""name"": ""title"",  
                            ""source"": ""/document/title""  
                        }},  
                        {{  
                            ""name"": ""loads"",  
                            ""source"": ""/document/loads""  
                        }},  
                        {{  
                            ""name"": ""header_1"",  
                            ""source"": ""/document/markdownDocument/*/sections/h1""  
                        }},  
                        {{  
                            ""name"": ""header_2"",  
                            ""source"": ""/document/markdownDocument/*/sections/h2""  
                        }},  
                        {{  
                            ""name"": ""header_3"",  
                            ""source"": ""/document/markdownDocument/*/sections/h3""  
                        }}  
                    ]  
                }}  
            ],  
            ""parameters"": {{  
                ""projectionMode"": ""skipIndexingParentDocuments""  
            }}  
        }}  
    }}";  
  
    using (var client = new HttpClient())  
    {  
        client.DefaultRequestHeaders.Add("api-key", searchApiKey);  
        var content = new StringContent(payload, Encoding.UTF8, "application/json");  
  
        HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
        Console.WriteLine("Response: {0}", response);  
        Console.WriteLine("HTTP Status Code: {0}", response.StatusCode);  
  
        if (!response.IsSuccessStatusCode)  
        {  
            Console.WriteLine("Error creating skillset: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
        }  
    }  
}  

In [97]:
CreateSkillset(azureSearchServiceName, azureSearchApiKey, azureSearchIndex, azureOpenAiEndpoint, azureOpenAiEmbeddingModelName, azureAiServicesEndpoint, azureOpenAiEmbeddingDimensions);

https://mmchrdemo03search.search.windows.net//skillsets/vector-manual31-skillset?api-version=2024-11-01-preview
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766FB4504F66"
  Location: https://mmchrdemo03search.search.windows.net:443/skillsets('vector-manual31-skillset')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: c5f81d8b-bac5-449c-baf7-ea4a67b8d153
  elapsed-time: 80
  Date: Tue, 08 Apr 2025 07:33:49 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


In [98]:
public static void CreateIndexer(string serviceName, string index, string searchKey)  
{  
    string endpoint = $"https://{serviceName}.search.windows.net/";  
    string url = $"{endpoint}/indexers/{index}-indexer/?api-version=2024-11-01-preview";  
    Console.WriteLine(url);  
  
    var payload = $@"  
    {{  
        ""name"": ""{index}-indexer"",  
        ""description"": null,  
        ""dataSourceName"": ""{index}-datasource"",  
        ""skillsetName"": ""{index}-skillset"",  
        ""targetIndexName"": ""{index}"",  
        ""disabled"": null,  
        ""schedule"": null,  
        ""parameters"": {{  
            ""batchSize"": null,  
            ""maxFailedItems"": null,  
            ""maxFailedItemsPerBatch"": null,  
            ""base64EncodeKeys"": null,  
            ""configuration"": {{  
                ""dataToExtract"": ""contentAndMetadata"",  
                ""parsingMode"": ""default"",  
                ""allowSkillsetToReadFileData"": true  
            }}  
        }},  
        ""fieldMappings"": [  
            {{  
                ""sourceFieldName"": ""metadata_storage_name"",  
                ""targetFieldName"": ""title"",  
                ""mappingFunction"": null  
            }}  
        ],  
        ""outputFieldMappings"": [],  
        ""cache"": null,  
        ""encryptionKey"": null  
    }}";  
  
    using (var client = new HttpClient())  
    {  
        client.DefaultRequestHeaders.Add("api-key", searchKey);  
        var content = new StringContent(payload, Encoding.UTF8, "application/json");  
  
        HttpResponseMessage response = client.PutAsync(url, content).GetAwaiter().GetResult();  
        Console.WriteLine("Response: {0}", response);  
        Console.WriteLine("HTTP Status Code: {0}", response.StatusCode);  
  
        if (!response.IsSuccessStatusCode)  
        {  
            Console.WriteLine("Error creating skillset: {0}", response.Content.ReadAsStringAsync().GetAwaiter().GetResult());  
        }  
    } 
}  

In [99]:
CreateIndexer(azureSearchServiceName, azureSearchIndex, azureSearchApiKey);

https://mmchrdemo03search.search.windows.net//indexers/vector-manual31-indexer/?api-version=2024-11-01-preview
Response: StatusCode: 201, ReasonPhrase: 'Created', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers:
{
  Transfer-Encoding: chunked
  ETag: "0x8DD766FB617C450"
  Location: https://mmchrdemo03search.search.windows.net:443/indexers('vector-manual31-indexer')?api-version=2024-11-01-preview
  Server: Microsoft-IIS/10.0
  Strict-Transport-Security: max-age=2592000
  Strict-Transport-Security: max-age=15724800; includeSubDomains
  Preference-Applied: odata.include-annotations="*"
  OData-Version: 4.0
  request-id: b688c364-a6af-404f-b31a-2550cb5994ee
  elapsed-time: 2281
  Date: Tue, 08 Apr 2025 07:33:52 GMT
  Content-Type: application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8
}
HTTP Status Code: Created


In [100]:
using System;
using Azure.AI.OpenAI;
using System.ClientModel;
using Azure.AI.OpenAI.Chat;
using OpenAI.Chat;
using static System.Environment;
using Newtonsoft.Json.Schema.Generation;

public class OpenAIResponseLoads()  
{  
        public required List<string> Loads { get; set; }  
    }

static string GetEnvironmentVariable(string key)
{
    return Environment.GetEnvironmentVariable(key) ?? throw new InvalidOperationException($"Environment variable '{key}' not found.");
}

// Create the clients
string endpoint = GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
string key = GetEnvironmentVariable("AZURE_OPENAI_API_KEY");


var openAIClient = new AzureOpenAIClient(new Uri(endpoint), new ApiKeyCredential(key));
var client = openAIClient.GetChatClient("gpt-4o");


var chat = new List<ChatMessage>
            {
                new SystemChatMessage("Objective: Extract unique load identifier from bills of lading and compile them into a distinct, structured list. A 'load' refers to the specific cargo or shipment. The output should exclude duplicates and irrelevant information. example of loads would be: L12345"),
                new UserChatMessage("BILL OF LADING (BOL) Document Number: BOL-2023-001234 Date of Issue: October 5, 2023 1. Load: L12345 2. Load: L67890 3. Load: L12345 4. Load: L54321 5. Load: L67890 6. Load: L11111 7. Load: L22222 8. Load: L33333 9. Load: L44444 10. Load: L55555"),
            };

            // Get the schema of the class for the structured response
JSchemaGenerator generator = new JSchemaGenerator();
var jsonSchema = generator.Generate(typeof(OpenAIResponseLoads)).ToString();

            // // Get a completion with structured output
            var chatUpdates = await client.CompleteChatAsync(
                chat,
                new ChatCompletionOptions
                {
                    ResponseFormat = ChatResponseFormat.CreateJsonSchemaFormat(
                        "openAIResponseLoads",
                        BinaryData.FromString(jsonSchema))
                });


// Deserialize the response to the defined class
var responseContent = chatUpdates.GetRawResponse().Content.ToString();

In [101]:
chatUpdates.Value.Content[0].Text

{"Loads":["L12345","L67890","L54321","L11111","L22222","L33333","L44444","L55555"]}

In [102]:
var data = JsonConvert.DeserializeObject<OpenAIResponseLoads>(chatUpdates.Value.Content[0].Text);
Console.WriteLine("Extracted Loads: " + string.Join(", ", data.Loads));


Extracted Loads: L12345, L67890, L54321, L11111, L22222, L33333, L44444, L55555


In [103]:
data.Loads

[ L12345, L67890, L54321, L11111, L22222, L33333, L44444, L55555 ]